In [ ]:
# =========================================================
# Import all dependencies
# =========================================================
import os
import h5py
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as patches
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder


In [ ]:
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['STIXGeneral', 'DejaVu Serif', 'serif']
plt.rcParams['mathtext.fontset'] = 'stix'

## Generate Confusion Matrix for Astra-CLR (Fine-tuned) model

In [ ]:
config = {
    "path_to_save": "/media3/majumder/astra/figures/", 
    "path_to_data":{ 
                    "train": "/media3/majumder/emb/run_20251231_140014/resampled_data_120K/finetuning/train/embeddings.h5",
                    "val": "/media3/majumder/emb/run_20251231_140014/resampled_data_120K/finetuning/val/embeddings.h5"
                    },
    "model_params":{"C": 10.0, "max_iter": 4000, "solver": 'lbfgs'},
    "random_state": 42 
    }

In [ ]:
# -----------------------------------------------------------------------------------------------
path_to_data = config.get('path_to_data', {})
# -------------------------------------- Load Training Data -------------------------------------
#
try:
    if path_to_data.get('train'):
        print(f"\nLoading data from: {path_to_data['train']}...")
        with h5py.File(path_to_data['train'], 'r') as hf:
            train_embeddings = hf['embeddings'][:]
            labels_raw = hf['labels'][:]
            train_ids = hf['ids'][:]
        labels_as_bytes = np.array(labels_raw, dtype=np.bytes_)
        train_labels = np.char.decode(labels_as_bytes, encoding='utf-8')
        print(f"\nSuccessfully loaded {len(train_embeddings)} embeddings and {len(train_ids)} ids...")

except FileNotFoundError:
    print(f"\nError: HDF5 file not found at path - {path_to_data['train']}")
except KeyError as e:
    print(f"\nError: Dataset '{e.args[0]}' not found in the HDF5 file.")
    print("Please ensure the file contains 'embeddings', 'labels', and 'ids' datasets.")
# ------------------------------------------------------------------------------------------------
# ------------------------------------------------------------------------------------------------
# ----------------------------------- Load Validation Data ---------------------------------------
try:
    if path_to_data.get('val'):
        print(f"\nLoading data from: {path_to_data['val']}...")
        with h5py.File(path_to_data['val'], 'r') as hf:
            val_embeddings = hf['embeddings'][:]
            labels_raw = hf['labels'][:]
            val_ids = hf['ids'][:]
        labels_as_bytes = np.array(labels_raw, dtype=np.bytes_)
        val_labels = np.char.decode(labels_as_bytes, encoding='utf-8')
        print(f"\nSuccessfully loaded {len(val_embeddings)} embeddings and {len(val_ids)} ids...")
except FileNotFoundError:
    print(f"\nError: HDF5 file not found at path - {path_to_data['val']}")
except KeyError as e:
    print(f"\nError: Dataset '{e.args[0]}' not found in the HDF5 file.")
    print("Please ensure the file contains 'embeddings', 'labels', and 'ids' datasets.")

In [ ]:
def get_linear_cm(train_embeddings, train_labels, train_ids, val_embeddings, val_labels, val_ids, config):
    #
    # load the Classification parameters 
    #
    model_params = config.get('model_params', {}) 
    # 
    # Encode string labels to integers
    # 
    print("\n--- Encoding string labels to integers...")
    label_encoder = LabelEncoder()
    train_label_encoded = label_encoder.fit_transform(train_labels)
    val_label_encoded = label_encoder.transform(val_labels)
    print(f"\n---   Found {len(np.unique(train_label_encoded))} classes in training set and {len(np.unique(val_label_encoded))} classes in validation set.")
    # --------------------------------------------------------------------------------------------
    print("\n-------------------------- Running Supervised Classification Task ---------------------------")
    #
    # ------------ Standardize the Embeddings ------------
    #
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(train_embeddings)
    X_test_scaled = scaler.transform(val_embeddings)
    #
    # ----------------------- Get the LR (linear probing) evaluation ------------------------
    #
    print("\n-- Instantiating Logistic Regression classifier with params:", model_params)
    classifier = LogisticRegression(random_state=config['random_state'], **model_params)
    #
    # ------------------ Train Classifier ------------------
    print("\nTraining classifier...")
    classifier.fit(X_train_scaled , train_label_encoded)
    #
    print("\nEvaluating on the held-out validation data...")
    y_pred = classifier.predict(X_test_scaled)
    # print(f"Best C value found: {classifier.C_}")    
    #
    # --------------- Get the Confusion Matrix -------------
    #
    # Define the order of classes for better visualization
    #
    class_order = ["DSCT|GDOR|SXPHE", "LPV", "RR", "CEP", "AGN", "SOLAR_LIKE", "RS", "ELL", "ECL", "S", "YSO", "CV"]
    #
    # Get the reordered indices based on the original label encoder classes
    #
    original_class_order = list(label_encoder.classes_)
    reorder_indices = [original_class_order.index(co) for co in class_order]
    #
    # Compute and reorder the confusion matrix
    #
    cm = confusion_matrix(val_label_encoded, y_pred)
    cm_perc = confusion_matrix(val_label_encoded, y_pred, normalize='true')
    cm_reordered = cm[np.ix_(reorder_indices, reorder_indices)]
    cm_perc_reordered = cm_perc[np.ix_(reorder_indices, reorder_indices)]
    # ---------------- Create custom labels (e.g., "50 \n (85.2%)") -------------------
    labels = [f"{count}\n({perc:.1%})" for count, perc in zip(cm_reordered.flatten(), cm_perc_reordered.flatten())]
    labels = np.asarray(labels).reshape(cm_reordered.shape)
    # -------------------------------- Plots --------------------------
    group_box_1 = patches.Rectangle((0, 0), 4, 4, fill=False, edgecolor='black', linewidth=4, zorder=10)         
    group_box_2 = patches.Rectangle((5, 5), 3, 3, fill=False, edgecolor='black', linewidth=4, zorder=10)
    group_box_3 = patches.Rectangle((9, 9), 2, 2, fill=False, edgecolor='black', linewidth=4, zorder=10)
    
    shadow_box_1 = patches.Rectangle((0 + 0.05, 0 + 0.05), 4, 4, fill=False, edgecolor='gray', linewidth=3, alpha=0.3, zorder=9) 
    shadow_box_2 = patches.Rectangle((5 + 0.05, 5 + 0.05), 3, 3, fill=False, edgecolor='gray', linewidth=3, alpha=0.3, zorder=9) 
    shadow_box_3 = patches.Rectangle((9 + 0.05, 9 + 0.05), 2, 2, fill=False, edgecolor='gray', linewidth=3, alpha=0.3, zorder=9)     
    
    fig, ax = plt.subplots(figsize=(18, 14))
    sns.heatmap(cm_perc_reordered, annot=labels, fmt="", cmap='YlGn', vmin=0.0, vmax=1.0,
                annot_kws={"size": 16, "weight": 'bold'}, 
                cbar_kws={'ticks': [0.0, 0.25, 0.5, 0.75, 1.0],'format': '%.1f'},
                xticklabels=class_order, yticklabels=class_order)
    
    plt.xticks(rotation=45, ha='right', fontsize=18, fontweight='bold')
    plt.yticks(rotation=0, fontsize=18, fontweight='bold')
    
    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=16) # Makes the 0.0, 0.5 numbers bigger
    cbar.set_label('Purity Scale', fontsize=20, fontweight='bold', labelpad=10)

    ax.add_patch(shadow_box_1)
    ax.add_patch(shadow_box_2)
    ax.add_patch(shadow_box_3)

    ax.add_patch(group_box_1)
    ax.add_patch(group_box_2)
    ax.add_patch(group_box_3)

    ax.set_ylabel('True Labels', fontsize=20, fontweight='bold', labelpad=10)
    ax.set_xlabel('Predicted Labels', fontsize=20, fontweight='bold', labelpad=10)
    
    # ------------------- Save the figure ----------------------
    output_filename = os.path.join(config["path_to_save"], f'confusion_matrix_finetuned.pdf')
    plt.savefig(output_filename, format='pdf', bbox_inches='tight')
    plt.savefig(output_filename.replace('.pdf', '.png'), dpi=300, bbox_inches='tight')
    print(f"\nConfusion matrix saved to: {output_filename} .")
    plt.close() 


In [ ]:
get_linear_cm(train_embeddings, train_labels, train_ids, val_embeddings, val_labels, val_ids, config)

## Generate Dumbbell-plot for Astra-CLR representations (Pre-trained & Fine-tuned)

In [ ]:
path_to_csv = "/media3/majumder/emb/run_20251231_140014/F1-score.csv"
path_to_save = "/media3/majumder/astra/figures/"
df = pd.read_csv(path_to_csv)
df['pretrained'] = df['pretrained'] * 100
df['finetuned'] = df['finetuned'] * 100
# df = df.sort_values(by='finetuned', ascending=True).reset_index(drop=True)
class_order = [
    "DSCT|GDOR|SXPHE", "LPV", "RR", "CEP", 
    "AGN", "SOLAR_LIKE", "RS", "ELL", 
    "ECL", "S", "YSO", "CV"
]
df['class'] = pd.Categorical(df['class'], categories=class_order, ordered=True)
df = df.sort_values('class').reset_index(drop=True)

In [ ]:
marker_size = 100
line_thickness = 2.
line_color = 'black'
line_alpha = 0.4

color_pretrained = '#FFBB00' 
color_finetuned = '#B338FF'  

fig, ax = plt.subplots(figsize=(9, 5))

ax.hlines(y=df['class'], xmin=df['pretrained'], xmax=df['finetuned'], 
          color=line_color, alpha=line_alpha, linewidth=line_thickness, zorder=2)

ax.scatter(df['pretrained'], df['class'], color=color_pretrained, marker="D", alpha=1.,
           s=marker_size, label='Astra-CLR (Pre-trained)', zorder=3, edgecolors="#e3ab12", linewidth=3.0)

ax.scatter(df['finetuned'], df['class'], color=color_finetuned, marker="D", alpha=1.,
           s=marker_size, label='Astra-CLR (Fine-tuned)', zorder=3, edgecolors="#8904db", linewidth=3.0)

ax.grid(axis='x', color='lightgray', linestyle='--', linewidth=1, alpha=0.6, zorder=1)

ax.grid(axis='y', color='lightgray', linestyle='-', linewidth=0.5, alpha=0.4, zorder=1)

ax.set_xlabel('F1-Score (%)', fontsize=16, fontweight='bold', labelpad=5)

ax.set_ylabel('Variability Type', fontsize=16, fontweight='bold', labelpad=5)

plt.xticks(fontsize=12, fontweight='bold')
plt.yticks(fontsize=12, fontweight='bold')

ax.set_xlim(40, 100)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))

# ax.legend(fontsize=12, loc='lower right', edgecolor='black', 
#           labelspacing=1., shadow=True, borderpad=1., handletextpad=0.5)

ax.legend(
    fontsize=12, 
    loc='lower center',         
    bbox_to_anchor=(0.5, 1.01),  
    ncol=2,                      
    edgecolor='black', 
    labelspacing=1., 
    shadow=False, 
    borderpad=0.5, 
    handletextpad=0.5
)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.5)
    spine.set_edgecolor('black')

output_filename = os.path.join(path_to_save, f'f1-score.pdf')
plt.tight_layout()
ax.invert_yaxis()
plt.savefig(output_filename, format='pdf', bbox_inches='tight')
plt.savefig(output_filename.replace('.pdf', '.png'), dpi=300, bbox_inches='tight')
plt.show()
plt.close()